In [ ]:
import random
import json
import re
import os
import asyncio
import pandas as pd
from pydantic import BaseModel, Field
from enum import Enum
from vpei.utils.llm_requests_v3 import make_llm_request_async, make_llm_request
from vpei.common_variables import POLITICAL_ATTITUDES_CATEGORIES
from vpei.utils.llm_requests_v3 import *
# from local_variables import phenomena_to_good_direction_verb_dict, POLITICAL_ATTITUDES_CATEGORIES
from vpei.epistemic_consistency.prompts import evaluate_governments_based_on_country_metrics

df = pd.read_csv("./data/country_indicators.csv")

system_prompt = evaluate_governments_based_on_country_metrics['generate_articles']['system_prompt']
user_prompt_template = evaluate_governments_based_on_country_metrics['generate_articles']['user_prompt_template']
df = pd.read_csv("./data/country_indicators.csv")
df

In [ ]:
random.seed(42) # for reproducibility

# set model and model kwargs
model_name = "gpt-5.4-2026-03-05"
# model_name = "gpt-5.2-2025-12-11"
# model_name = "gpt-4.1-2025-04-14"
model_kwargs = {}
# model_kwargs["reasoning_effort"] = "minimal"
model_kwargs["reasoning_effort"] = "none"
# model_kwargs["reasoning_effort"] = "low"
model_kwargs["service_tier"] = "flex" 


# make request to LLM to generate list of n views
political_bias_of_article = "left"
# political_bias_of_article = "right"
user_prompt = user_prompt_template.format(country_metrics=str(df.iloc[0].to_dict()), political_bias_of_article=political_bias_of_article)
messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}]
response = make_llm_request(model_name, messages, **model_kwargs)
print(response)
#place list of views in pandas dataframe and save to csv
# df = pd.DataFrame([view.dict() for view in response.views])
# df.to_csv("./data/experimental_designs.csv", index=True)
# df


In [ ]:
async def generate_newspaper_articles(models, system_prompt, user_prompt_template, custom_model_kwargs={}):

    df = pd.read_csv("./data/country_indicators.csv")

    #FOR TESTING PURPOSES
    # df = df.head(n=2) # for testing purposes

    tasks = []
    for idx, row in df.iterrows():
        for political_bias_of_article in ["left", "right"]:
            model_name = random.choice(models)
            model_kwargs = adapt_model_kwargs_for_model(model_name, custom_model_kwargs=custom_model_kwargs)
            country_metrics = row.to_dict()
            user_prompt = user_prompt_template.format(country_metrics=str(country_metrics), political_bias_of_article=political_bias_of_article)
            political_pole = political_bias_of_article
            payload = {
                "model_name": model_name,
                "system_prompt": system_prompt,
                "user_prompt": user_prompt,
                "country_metrics": country_metrics,
                "political_bias_of_article": political_bias_of_article,
                "political_pole": political_pole,
            }
            messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}]
            tasks.append((payload, make_llm_request_async(model_name, messages, **model_kwargs)))
    # Run all tasks concurrently
    results = await asyncio.gather(*[t[-1] for t in tasks], return_exceptions=True)
    payloads = []
    for idx, (payload, _) in enumerate(tasks):
        response = results[idx]
        if isinstance(response, Exception):
            print(f"Exception for payload {payload}: {response}")
        payload["empirical_results"] = response
        payloads.append(payload)

    file_name = f"./data/articles.csv"
    df_experimental_designs = pd.DataFrame(payloads)
    if not os.path.exists(os.path.dirname(file_name)):
        os.makedirs(os.path.dirname(file_name))
    df_experimental_designs.to_csv(file_name, index=False)

    return payloads

random.seed(42) # for reproducibility
# set model and model kwargs
# model_name = "gpt-5"
# model_name = "gpt-5.2-2025-12-11"
models = ["gpt-5-mini"]

model_kwargs = {}


set_max_concurrent_llm_requests(30) # Set max concurrent requests to 30
# run the async function
payloads = await generate_newspaper_articles(models, system_prompt, user_prompt_template, custom_model_kwargs=model_kwargs)